# 04 — Experiment grid

Runs the factorial defined in the protocol:

`{original, improved} x {random_70_30, day_ordered} x {attempted policy} x {logreg, rf, xgboost} x 5 seeds`

plus the Arm B matched-subset runs.

**Checkpointing:** every completed configuration is appended to
`results/runs.csv` and skipped on re-run. Colab disconnects are expected and harmless.

No hyperparameter tuning. Identical settings across dataset versions — that is the whole point.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os
DRIVE_ROOT = '/content/drive/MyDrive/research/ids-label-correction'
sys.path.insert(0, os.path.join(DRIVE_ROOT, 'src'))

import importlib, config as C, helpers as H
importlib.reload(C); importlib.reload(H)

import pandas as pd, numpy as np, json, time, itertools, hashlib
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (f1_score, balanced_accuracy_score,
                             matthews_corrcoef, accuracy_score,
                             classification_report)
from xgboost import XGBClassifier

Mounted at /content/drive


In [2]:
def make_model(name, seed):
    if name == 'logreg':
        return LogisticRegression(max_iter=1000, n_jobs=-1, random_state=seed)
    if name == 'random_forest':
        return RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=seed)
    if name == 'xgboost':
        return XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.3,
                             tree_method='hist', n_jobs=-1, random_state=seed,
                             eval_metric='logloss')
    raise ValueError(name)


def apply_attempted(df, policy):
    """H3 knob. The 'attempted category' column uses a sentinel, not nulls,
    so detection MUST come from the label string (same fix as notebook 02)."""
    att = H.is_attempted(df['label'])
    if not att.any():
        return df
    if policy == 'as_attack':
        return df
    if policy == 'as_benign':
        df = df.copy()
        df.loc[att, 'label'] = 'BENIGN'
        return df
    if policy == 'dropped':
        return df.loc[~att].copy()
    raise ValueError(policy)


def split(df, protocol, seed):
    if protocol == 'random_70_30':
        idx_tr, idx_te = train_test_split(
            df.index, test_size=0.30, random_state=seed,
            stratify=H.binarise(df['label']))
        return df.loc[idx_tr], df.loc[idx_te]
    if protocol == 'day_ordered':
        tr = df[df['day'].isin(C.TRAIN_DAYS)]
        te = df[df['day'].isin(C.TEST_DAYS)]
        return tr, te
    raise ValueError(protocol)

In [3]:
RUNS_PATH = os.path.join(C.RESULTS, 'runs.csv')

# Fixed superset schema. Every append writes every column, so the file can
# never again fork into per-arm shapes. (The original append wrote each
# record's own fields; pandas then padded short rows POSITIONALLY, silently
# shifting Arm B's metrics two columns left — balanced_acc was displayed as
# macro_f1. A schema drift corrupting reported metrics, inside a study about
# silent data corruption. It stays in the log as a lesson.)
ALL_COLS = ['run_id', 'arm', 'task', 'version', 'split', 'attempted', 'model',
            'seed', 'n_train', 'n_test', 'n_features', 'duplicates_removed',
            'fit_seconds', 'macro_f1', 'weighted_f1', 'balanced_acc', 'mcc',
            'accuracy', 'attack_recall', 'attack_precision', 'benign_fpr']

# The three row shapes that exist in the wild, by field count:
LEGACY_SCHEMAS = {
    17: ['run_id', 'arm', 'version', 'split', 'attempted', 'model', 'seed',
         'n_train', 'n_test', 'n_features', 'duplicates_removed', 'fit_seconds',
         'macro_f1', 'weighted_f1', 'balanced_acc', 'mcc', 'accuracy'],
    15: ['run_id', 'arm', 'version', 'split', 'attempted', 'model', 'seed',
         'n_train', 'n_test', 'n_features',
         'macro_f1', 'weighted_f1', 'balanced_acc', 'mcc', 'accuracy'],
    20: ['run_id', 'arm', 'task', 'version', 'split', 'attempted', 'model',
         'seed', 'n_train', 'n_test', 'n_features', 'fit_seconds',
         'macro_f1', 'weighted_f1', 'balanced_acc', 'mcc', 'accuracy',
         'attack_recall', 'attack_precision', 'benign_fpr'],
}

import csv, shutil

def repair_runs_csv():
    """One-time migration to the canonical schema. Safe to re-run."""
    if not os.path.exists(RUNS_PATH):
        print('no runs.csv yet — nothing to repair')
        return
    with open(RUNS_PATH) as f:
        lines = [ln.rstrip('\n') for ln in f if ln.strip()]
    if lines and lines[0].split(',') == ALL_COLS:
        print('runs.csv already canonical —', len(lines) - 1, 'rows')
        return
    backup = os.path.join(C.RESULTS, 'runs_backup_raw.csv')
    shutil.copy(RUNS_PATH, backup)
    recs, bad = [], 0
    for ln in lines[1:]:
        parts = ln.split(',')
        sch = LEGACY_SCHEMAS.get(len(parts))
        if sch is None:
            bad += 1
            continue
        rec = dict(zip(sch, parts))
        rec.setdefault('task', 'multiclass')
        recs.append(rec)
    with open(RUNS_PATH, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=ALL_COLS)
        w.writeheader()
        for r in recs:
            w.writerow({c: r.get(c, '') for c in ALL_COLS})
    print(f'repaired runs.csv: {len(recs)} rows migrated, {bad} unparseable, '
          f'raw backup at {backup}')

repair_runs_csv()


def load_runs():
    if os.path.exists(RUNS_PATH):
        return pd.read_csv(RUNS_PATH)
    return pd.DataFrame()


def run_id(**kw):
    return hashlib.md5(json.dumps(kw, sort_keys=True).encode()).hexdigest()[:12]


def append_run(rec):
    new = not os.path.exists(RUNS_PATH)
    with open(RUNS_PATH, 'a', newline='') as f:
        w = csv.DictWriter(f, fieldnames=ALL_COLS)
        if new:
            w.writeheader()
        w.writerow({c: rec.get(c, '') for c in ALL_COLS})


def evaluate(y_true, y_pred):
    return {
        'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'weighted_f1': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'balanced_acc': balanced_accuracy_score(y_true, y_pred),
        'mcc': matthews_corrcoef(y_true, y_pred),
        'accuracy': accuracy_score(y_true, y_pred),
    }

runs.csv already canonical — 275 rows


In [4]:
def prepare(version, attempted_policy, dedup_flag=True):
    """Load, apply the attempted policy, clean, optionally dedup. Returns df + report."""
    path = os.path.join(C.INTERIM, f'{version}.parquet')
    df = pd.read_parquet(path)
    df = apply_attempted(df, attempted_policy)
    df, rep = H.clean_features(df)
    if dedup_flag:
        df, rep2 = H.dedup(df)
        rep.update(rep2)
    df['y'] = H.coarse_class(df['label'])
    return df, rep


def align_features(tr, te):
    """Both versions must use only columns they share with themselves across splits."""
    feats = [c for c in H.feature_columns(tr) if c in te.columns and c != 'y']
    feats = [c for c in feats if pd.api.types.is_numeric_dtype(tr[c])]
    return feats

In [5]:
# ---- ARM A: dataset-level grid ------------------------------------------
done = load_runs()
done_ids = set(done['run_id']) if len(done) else set()

grid = list(itertools.product(
    ['original', 'improved'],
    C.SPLIT_PROTOCOLS,
    C.ATTEMPTED_POLICIES,
    C.MODELS,
    C.SEEDS,
))
print(len(grid), 'configurations')

cache = {}
for version, proto, policy, model_name, seed in grid:
    # attempted policy is meaningless for the original version — run it once
    if version == 'original' and policy != 'as_attack':
        continue

    rid = run_id(arm='A', version=version, split=proto,
                 attempted=policy, model=model_name, seed=seed)
    if rid in done_ids:
        continue

    ck = (version, policy)
    if ck not in cache:
        cache.clear()
        cache[ck] = prepare(version, policy)
    df, rep = cache[ck]

    tr, te = split(df, proto, seed)
    feats = align_features(tr, te)

    Xtr, ytr = tr[feats].values, tr['y'].values
    Xte, yte = te[feats].values, te['y'].values

    if model_name == 'logreg':
        sc = StandardScaler().fit(Xtr)
        Xtr, Xte = sc.transform(Xtr), sc.transform(Xte)

    classes = sorted(set(ytr))
    cmap = {c: i for i, c in enumerate(classes)}
    ytr_i = np.array([cmap[v] for v in ytr])
    mask = np.isin(yte, classes)
    yte_i = np.array([cmap[v] for v in yte[mask]])

    t0 = time.time()
    m = make_model(model_name, seed).fit(Xtr, ytr_i)
    pred = m.predict(Xte[mask])
    res = evaluate(yte_i, pred)

    append_run({
        'run_id': rid, 'arm': 'A', 'version': version, 'split': proto,
        'attempted': policy, 'model': model_name, 'seed': seed,
        'n_train': len(tr), 'n_test': int(mask.sum()), 'n_features': len(feats),
        'duplicates_removed': rep.get('exact_duplicates_removed', 0),
        'fit_seconds': round(time.time() - t0, 1), **res,
    })
    print(f'{version:9s} {proto:13s} {policy:10s} {model_name:14s} seed={seed:3d} '
          f'macroF1={res["macro_f1"]:.4f}')

print('\nArm A complete')

180 configurations

Arm A complete


In [6]:
# ---- ARM B: matched subset, label is the only difference ----------------
matched = pd.read_parquet(os.path.join(C.INTERIM, 'matched.parquet'))
done = load_runs()
done_ids = set(done['run_id']) if len(done) else set()

base, rep_m = H.clean_features(matched)

for label_source in ['label_original', 'label_improved']:
    for proto in C.SPLIT_PROTOCOLS:
        for model_name in C.MODELS:
            for seed in C.SEEDS:
                rid = run_id(arm='B', labels=label_source, split=proto,
                             model=model_name, seed=seed)
                if rid in done_ids:
                    continue

                df = base.copy()
                df['label'] = df[label_source]
                df['y'] = H.coarse_class(df['label'])

                tr, te = split(df, proto, seed)
                feats = [c for c in align_features(tr, te)
                         if c not in ('label_original', 'label_improved',
                                      'attempted', 'label')]

                Xtr, ytr = tr[feats].values, tr['y'].values
                Xte, yte = te[feats].values, te['y'].values

                if model_name == 'logreg':
                    sc = StandardScaler().fit(Xtr)
                    Xtr, Xte = sc.transform(Xtr), sc.transform(Xte)

                classes = sorted(set(ytr))
                cmap = {c: i for i, c in enumerate(classes)}
                ytr_i = np.array([cmap[v] for v in ytr])
                mask = np.isin(yte, classes)
                yte_i = np.array([cmap[v] for v in yte[mask]])

                m = make_model(model_name, seed).fit(Xtr, ytr_i)
                pred = m.predict(Xte[mask])
                res = evaluate(yte_i, pred)

                append_run({
                    'run_id': rid, 'arm': 'B', 'version': label_source,
                    'split': proto, 'attempted': 'n/a', 'model': model_name,
                    'seed': seed, 'n_train': len(tr), 'n_test': int(mask.sum()),
                    'n_features': len(feats), **res,
                })
                print(f'B {label_source:15s} {proto:13s} {model_name:14s} '
                      f'seed={seed:3d} macroF1={res["macro_f1"]:.4f}')

print('\nArm B complete')


Arm B complete


In [7]:
# ---- ARM A (binary): benign vs attack ---------------------------------------
# Multiclass day-ordered is structurally degenerate on CICIDS2017: attack
# families are day-segregated (Tue brute force, Wed DoS, Thu web/infiltration,
# Fri bot/portscan/DDoS), so Thu-Fri test families never appear in Mon-Wed
# training; after masking, the test collapses toward one class and macro-F1
# pins at 1/k. This arm asks the temporal question at the granularity the
# dataset supports: does a model trained on Mon-Wed flag Thu-Fri's UNSEEN
# attack families as attacks at all? (This is the zero-day setting.)
# Multiclass random-split runs above remain valid and are kept.

from sklearn.metrics import recall_score, precision_score

def eval_binary(y_true, y_pred):
    return {
        'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'weighted_f1': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'balanced_acc': balanced_accuracy_score(y_true, y_pred),
        'mcc': matthews_corrcoef(y_true, y_pred),
        'accuracy': accuracy_score(y_true, y_pred),
        'attack_recall': recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        'attack_precision': precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        'benign_fpr': 1.0 - recall_score(y_true, y_pred, pos_label=0, zero_division=0),
    }

done = load_runs()
done_ids = set(done['run_id']) if len(done) else set()

cache = {}
for version, proto, policy, model_name, seed in grid:
    if version == 'original' and policy != 'as_attack':
        continue

    rid = run_id(arm='A', task='binary', version=version, split=proto,
                 attempted=policy, model=model_name, seed=seed)
    if rid in done_ids:
        continue

    ck = (version, policy)
    if ck not in cache:
        cache.clear()
        cache[ck] = prepare(version, policy)
    df, rep = cache[ck]

    tr, te = split(df, proto, seed)
    feats = align_features(tr, te)

    ytr = H.binarise(tr['label']).values
    yte = H.binarise(te['label']).values
    Xtr, Xte = tr[feats].values, te[feats].values

    if model_name == 'logreg':
        sc = StandardScaler().fit(Xtr)
        Xtr, Xte = sc.transform(Xtr), sc.transform(Xte)

    t0 = time.time()
    m = make_model(model_name, seed).fit(Xtr, ytr)
    pred = m.predict(Xte)
    res = eval_binary(yte, pred)

    append_run({
        'run_id': rid, 'arm': 'A', 'task': 'binary', 'version': version,
        'split': proto, 'attempted': policy, 'model': model_name, 'seed': seed,
        'n_train': len(tr), 'n_test': len(te), 'n_features': len(feats),
        'fit_seconds': round(time.time() - t0, 1), **res,
    })
    print(f'BIN {version:9s} {proto:13s} {policy:10s} {model_name:14s} '
          f'seed={seed:3d} macroF1={res["macro_f1"]:.4f} '
          f'att-recall={res["attack_recall"]:.4f}')

print('\nArm A binary complete')

BIN improved  day_ordered   as_benign  random_forest  seed= 11 macroF1=0.6416 att-recall=0.2898
BIN improved  day_ordered   as_benign  random_forest  seed= 23 macroF1=0.6348 att-recall=0.2800
BIN improved  day_ordered   as_benign  random_forest  seed= 37 macroF1=0.6402 att-recall=0.2877
BIN improved  day_ordered   as_benign  random_forest  seed= 51 macroF1=0.6314 att-recall=0.2752
BIN improved  day_ordered   as_benign  random_forest  seed= 73 macroF1=0.6411 att-recall=0.2891
BIN improved  day_ordered   as_benign  xgboost        seed= 11 macroF1=0.5114 att-recall=0.1236
BIN improved  day_ordered   as_benign  xgboost        seed= 23 macroF1=0.5114 att-recall=0.1236
BIN improved  day_ordered   as_benign  xgboost        seed= 37 macroF1=0.5114 att-recall=0.1236
BIN improved  day_ordered   as_benign  xgboost        seed= 51 macroF1=0.5114 att-recall=0.1236
BIN improved  day_ordered   as_benign  xgboost        seed= 73 macroF1=0.5114 att-recall=0.1236
BIN improved  day_ordered   dropped    l

In [8]:
# ---- ARM B (binary): matched subset, benign vs attack -----------------------
matched = pd.read_parquet(os.path.join(C.INTERIM, 'matched.parquet'))
done = load_runs()
done_ids = set(done['run_id']) if len(done) else set()

base, rep_m = H.clean_features(matched)

for label_source in ['label_original', 'label_improved']:
    for proto in C.SPLIT_PROTOCOLS:
        for model_name in C.MODELS:
            for seed in C.SEEDS:
                rid = run_id(arm='B', task='binary', labels=label_source,
                             split=proto, model=model_name, seed=seed)
                if rid in done_ids:
                    continue

                df = base.copy()
                df['label'] = df[label_source]

                tr, te = split(df, proto, seed)
                feats = [c for c in align_features(tr, te)
                         if c not in ('label_original', 'label_improved',
                                      'attempted', 'label')]

                ytr = H.binarise(tr['label']).values
                yte = H.binarise(te['label']).values
                Xtr, Xte = tr[feats].values, te[feats].values

                if model_name == 'logreg':
                    sc = StandardScaler().fit(Xtr)
                    Xtr, Xte = sc.transform(Xtr), sc.transform(Xte)

                m = make_model(model_name, seed).fit(Xtr, ytr)
                pred = m.predict(Xte)
                res = eval_binary(yte, pred)

                append_run({
                    'run_id': rid, 'arm': 'B', 'task': 'binary',
                    'version': label_source, 'split': proto,
                    'attempted': 'n/a', 'model': model_name, 'seed': seed,
                    'n_train': len(tr), 'n_test': len(te),
                    'n_features': len(feats), **res,
                })
                print(f'BIN B {label_source:15s} {proto:13s} {model_name:14s} '
                      f'seed={seed:3d} macroF1={res["macro_f1"]:.4f} '
                      f'att-recall={res["attack_recall"]:.4f}')

print('\nArm B binary complete')

BIN B label_original  random_70_30  logreg         seed= 11 macroF1=0.9655 att-recall=0.9865
BIN B label_original  random_70_30  logreg         seed= 23 macroF1=0.9645 att-recall=0.9866
BIN B label_original  random_70_30  logreg         seed= 37 macroF1=0.9647 att-recall=0.9868
BIN B label_original  random_70_30  logreg         seed= 51 macroF1=0.9649 att-recall=0.9866
BIN B label_original  random_70_30  logreg         seed= 73 macroF1=0.9650 att-recall=0.9865
BIN B label_original  random_70_30  random_forest  seed= 11 macroF1=0.9999 att-recall=0.9998
BIN B label_original  random_70_30  random_forest  seed= 23 macroF1=0.9999 att-recall=0.9997
BIN B label_original  random_70_30  random_forest  seed= 37 macroF1=0.9999 att-recall=0.9999
BIN B label_original  random_70_30  random_forest  seed= 51 macroF1=0.9999 att-recall=0.9997
BIN B label_original  random_70_30  random_forest  seed= 73 macroF1=0.9999 att-recall=0.9999
BIN B label_original  random_70_30  xgboost        seed= 11 macroF1=0.

In [9]:
runs = load_runs()
runs = runs.drop_duplicates(subset='run_id', keep='last')
if 'task' not in runs.columns:
    runs['task'] = 'multiclass'
runs['task'] = runs['task'].fillna('multiclass')
print(len(runs), 'unique runs')

print('\nVALID configurations (multiclass random + binary both splits):')
valid = runs[(runs['task'] == 'binary') |
             ((runs['task'] == 'multiclass') & (runs['split'] == 'random_70_30'))]
display(valid.groupby(['task', 'arm', 'version', 'split', 'model'])['macro_f1']
        .agg(['mean', 'std', 'count']).round(4))

print('\nEXCLUDED as degenerate (multiclass day_ordered — attack families are')
print('day-segregated, so the masked test collapses toward one class):')
deg = runs[(runs['task'] == 'multiclass') & (runs['split'] == 'day_ordered')]
display(deg.groupby(['arm', 'version', 'model'])['macro_f1']
        .agg(['mean', 'std', 'count']).round(4))

360 unique runs

VALID configurations (multiclass random + binary both splits):


mean     std  \
task       arm version        split        model                           
binary     A   improved       day_ordered  logreg         0.6466  0.0014   
                                           random_forest  0.6377  0.0031   
                                           xgboost        0.5805  0.0527   
                              random_70_30 logreg         0.9896  0.0021   
                                           random_forest  0.9999  0.0000   
                                           xgboost        1.0000  0.0000   
               original       day_ordered  logreg         0.7401  0.0000   
                                           random_forest  0.6617  0.0068   
                                           xgboost        0.4393  0.0000   
                              random_70_30 logreg         0.8993  0.0009   
                                           random_forest  0.9994  0.0000   
                                           xgboost        0.9997  0.0000   
           B   label_improved day_ordered  logreg         0.6443  0.0000   
                                           random_forest  0.4053  0.0029   
                                           xgboost        0.4529  0.0000   
                              random_70_30 logreg         0.9910  0.0002   
                                           random_forest  0.9999  0.0000   
                                           xgboost        0.9999  0.0000   
               label_original day_ordered  logreg         0.5610  0.0000   
                                           random_forest  0.4340  0.0000   
                                           xgboost        0.4572  0.0000   
                              random_70_30 logreg         0.9649  0.0004   
                                           random_forest  0.9999  0.0000   
                                           xgboost        0.9999  0.0000   
multiclass A   improved       random_70_30 logreg         0.8764  0.0677   
                                           random_forest  0.9894  0.0068   
                                           xgboost        0.9691  0.0377   
               original       random_70_30 logreg         0.6671  0.0207   
                                           random_forest  0.9641  0.0253   
                                           xgboost        0.8524  0.0926   
           B   label_improved random_70_30 logreg         0.7420  0.0395   
                                           random_forest  0.9958  0.0019   
                                           xgboost        0.8745  0.1853   
               label_original random_70_30 logreg         0.6906  0.0239   
                                           random_forest  0.8679  0.0094   
                                           xgboost        0.7580  0.0962   

                                                          count  
task       arm version        split        model                 
binary     A   improved       day_ordered  logreg            15  
                                           random_forest     15  
                                           xgboost           15  
                              random_70_30 logreg            15  
                                           random_forest     15  
                                           xgboost           15  
               original       day_ordered  logreg             5  
                                           random_forest      5  
                                           xgboost            5  
                              random_70_30 logreg             5  
                                           random_forest      5  
                                           xgboost            5  
           B   label_improved day_ordered  logreg             5  
                                           random_forest      5  
                                           xgboost            5  
                              random_70_30 logreg


EXCLUDED as degenerate (multiclass day_ordered — attack families are
day-segregated, so the masked test collapses toward one class):


mean     std  count
arm version        model                               
A   improved       logreg         0.2777  0.0406     15
                   random_forest  0.4556  0.0763     15
                   xgboost        0.3333  0.0000     15
    original       logreg         0.3299  0.0000      5
                   random_forest  0.3666  0.0745      5
                   xgboost        0.3332  0.0000      5
B   label_improved logreg         0.3331  0.0000      5
                   random_forest  0.3333  0.0000      5
                   xgboost        0.3333  0.0000      5
    label_original logreg         0.3329  0.0000      5
                   random_forest  0.4000  0.0913      5
                   xgboost        0.3333  0.0000      5

## If Colab disconnects

Just re-run the two grid cells. Completed configurations are skipped via `run_id`.

Do not delete `results/runs.csv` to "start clean" — it is the experiment log, and
regenerating it silently is exactly the practice this study criticises.